# AlphaFlow v2 波动率包络策略 — 回测报告

**策略**: AlphaFlow v2  
**合约**: BTCUSDT.BINANCE 1m → BarGenerator合成1h  
**区间**: 2020-01-01 ~ 2026-05-30  
**参数**: vol_multiplier=3.0, filter_ema_period=0, use_dynamic_size=true, position_multiplier=0.005  
**仓位**: 动态 capital/ATR × 0.005

In [1]:
import sys
from pathlib import Path
PROJECT_DIR = Path("/home/admin/Desktop/hermes vnpy cta research/cta_harness")
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / "strategies"))

from datetime import datetime
import pandas as pd, numpy as np, json
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly.offline import iplot, init_notebook_mode
init_notebook_mode(connected=True)

from vnpy.trader.constant import Interval
from vnpy_ctastrategy.backtesting import BacktestingEngine
from vnpy_ctastrategy.base import BacktestingMode
from alpha_flow_strategy import AlphaFlowStrategy

print("环境就绪")

环境就绪


In [2]:
# 运行回测
engine = BacktestingEngine()
engine.set_parameters(
    vt_symbol="BTCUSDT.BINANCE",
    interval=Interval.MINUTE,
    start=datetime(2020, 1, 1),
    end=datetime(2026, 5, 30),
    rate=0.0004, slippage=0, size=1, pricetick=0.01, capital=200000,
    mode=BacktestingMode.BAR,
)
engine.add_strategy(AlphaFlowStrategy, {
    "vol_multiplier": 3.0, "filter_ema_period": 0,
    "use_dynamic_size": True, "position_multiplier": 0.005,
    "bar_interval_minutes": 60,
})
engine.load_data()
engine.run_backtesting()
df = engine.calculate_result()
stats = engine.calculate_statistics(output=False)

print(f"Sharpe={stats['sharpe_ratio']:.2f}  收益={stats['total_return']:.1f}%  MaxDD={stats['max_ddpercent']:.1f}%  交易={stats['total_trade_count']}")

2026-05-31 22:22:03.276930	开始加载历史数据
2026-05-31 22:22:03.280762	加载进度：# [0%]


2026-05-31 22:22:11.280930	加载进度：# [10%]


2026-05-31 22:22:19.417771	加载进度：## [20%]


2026-05-31 22:22:27.648399	加载进度：### [30%]


2026-05-31 22:22:35.656987	加载进度：#### [40%]


...

2026-05-31 22:23:37.563780	回放进度：====== [50%]


2026-05-31 22:23:38.416275	回放进度：======= [60%]


2026-05-31 22:23:39.302179	回放进度：======== [70%]


2026-05-31 22:23:40.190193	回放进度：========= [80%]


2026-05-31 22:23:41.072264	回放进度：========== [90%]
2026-05-31 22:23:41.078807	回放进度：=========== [100%]
2026-05-31 22:23:41.078873	历史数据回放结束
2026-05-31 22:23:41.078979	开始计算逐日盯市盈亏
2026-05-31 22:23:41.096609	逐日盯市盈亏计算完成
2026-05-31 22:23:41.097148	开始计算策略统计指标
2026-05-31 22:23:41.110837	策略统计指标计算完成
Sharpe=1.00  收益=160.6%  MaxDD=-11.8%  交易=520


In [3]:
# vnpy 官方四图
fig = engine.show_chart()
fig.update_layout(title_text="AlphaFlow v2 — BTCUSDT 1h 回测 (2020-2026)", title_font_size=16)
iplot(fig)

In [4]:
# 年度收益
df['year'] = pd.DatetimeIndex(df.index).year
yearly = df.groupby('year').agg(
    ret=('balance', lambda x: (x.iloc[-1]/x.iloc[0]-1)*100),
    dd=('drawdown', 'min')
).round(1)

fig_y = make_subplots(1, 2, subplot_titles=["年度收益 (%)", "最大回撤 (%)"])
fig_y.add_trace(go.Bar(x=yearly.index, y=yearly['ret'], marker_color='steelblue'), 1, 1)
fig_y.add_trace(go.Bar(x=yearly.index, y=-yearly['dd'], marker_color='coral'), 1, 2)
fig_y.update_layout(title="年度表现", height=400)
iplot(fig_y)
print(yearly.to_string())

       ret       dd
year               
2020  50.3 -29458.9
2021   6.9 -10674.2
2022  -2.7 -26252.2
2023  47.2 -37339.2
2024  13.3 -18802.8
2025   1.9 -16260.5
2026  -2.0 -23905.9


In [5]:
# 滚动 Sharpe
df['daily_ret'] = df['balance'].pct_change()
df['rolling_sharpe'] = df['daily_ret'].rolling(252).apply(
    lambda x: np.sqrt(252)*x.mean()/x.std() if x.std()>0 else 0
)
fig_r = go.Figure()
fig_r.add_trace(go.Scatter(x=df.index, y=df['rolling_sharpe'], name='Sharpe', line=dict(color='green')))
fig_r.add_hline(y=1, line_dash="dash", line_color="gray")
fig_r.add_hline(y=0, line_dash="dash", line_color="red")
fig_r.update_layout(title="滚动 Sharpe (252天)", height=400)
iplot(fig_r)

In [6]:
# 月度热力图
df['month'] = pd.DatetimeIndex(df.index).month
monthly = df.groupby(['year','month'])['daily_ret'].sum().unstack()

fig_h = go.Figure(go.Heatmap(
    z=monthly.values*100, x=[f'{m}月' for m in monthly.columns], y=monthly.index,
    colorscale='RdYlGn', zmid=0,
    text=[[f'{v:.1f}%' if not np.isnan(v) else '' for v in row] for row in (monthly.values*100)],
    texttemplate='%{text}',
))
fig_h.update_layout(title="月度收益率热力图 (%)", height=500)
iplot(fig_h)

In [7]:
# 完整统计
print("\n" + "="*55)
print("AlphaFlow v2 完整回测统计")
print("="*55)
for k, v in stats.items():
    if isinstance(v, float):
        print(f"  {k:28s}: {v:>12.2f}")
    else:
        print(f"  {k:28s}: {str(v):>12s}")
print("="*55)

with open("alpha_flow_stats.json","w") as f:
    json.dump(stats, f, indent=2, ensure_ascii=False, default=str)


AlphaFlow v2 完整回测统计
  start_date                  :   2020-01-01
  end_date                    :   2026-05-30
  total_days                  :         2342
  profit_days                 :          363
  loss_days                   :          418
  capital                     :       200000
  end_balance                 :    521166.05
  max_drawdown                :    -37339.17
  max_ddpercent               :       -11.82
  max_drawdown_duration       :           98
  total_net_pnl               :    321166.05
  daily_net_pnl               :       137.13
  total_commission            :     26216.03
  daily_commission            :        11.19
  total_slippage              :            0
  daily_slippage              :         0.00
  total_turnover              :  65540070.84
  daily_turnover              :     27984.66
  total_trade_count           :          520
  daily_trade_count           :         0.22
  total_return                :       160.58
  annual_return               :   